In [13]:
import pandas as pd
import numpy as np
import math
import os

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1 = math.radians(lat1); phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1); dl = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dl/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c


In [14]:
base_datos = "data"
mun_path_csv = "datos_municipios.csv"
cand_path_csv = "candidatos.csv"



municipios = pd.read_csv(mun_path_csv)
candidatos = pd.read_csv(cand_path_csv)


municipios.head(), candidatos.head()


(   cod_ine                 municipio provincia  poblacion    latitud  longitud
 0     5007  ALDEANUEVA DE SANTA CRUZ     ÁVILA        102  40.381960 -5.419593
 1     5008                 ALDEASECA     ÁVILA        203  41.049202 -4.817519
 2     5016                   ARÉVALO     ÁVILA       7830  41.065505 -4.718925
 3     5034  BLASCONUÑO DE MATACABRAS     ÁVILA         14  41.123872 -4.989882
 4     5035              BLASCOSANCHO     ÁVILA        100  40.878249 -4.637347,
     id_candidato                                          nombre      tipo  \
 0  AV_SONSESOLES             Hospital Nuestra Señora de Sonsoles  hospital   
 1        BU_HUBU         Hospital Universitario de Burgos (HUBU)  hospital   
 2      LE_HULEON      Complejo Asistencial Universitario de León  hospital   
 3      LE_BIERZO                             Hospital del Bierzo  hospital   
 4       PA_CAUPA  Complejo Asistencial Universitario de Palencia  hospital   
 
     municipio provincia  latitud  longitud

In [15]:
mun_cols = municipios.columns.str.lower()
cand_cols = candidatos.columns.str.lower()

def find_column(cols, names):
    for name in names:
        for c in cols:
            if c == name:
                return c
    return None

# Detectar columnas municipios
mun_id  = find_column(mun_cols, ['id'])
mun_nom = find_column(mun_cols, ['nombre','municipio'])
mun_lat = find_column(mun_cols, ['lat','latitude'])
mun_lon = find_column(mun_cols, ['lon','long'])
mun_pob = find_column(mun_cols, ['pob','poblacion'])

# Detectar columnas candidatos
cand_id  = find_column(cand_cols, ['id'])
cand_nom = find_column(cand_cols, ['nombre','municipio'])
cand_lat = find_column(cand_cols, ['lat'])
cand_lon = find_column(cand_cols, ['lon'])

municipios = municipios.rename(columns={
    mun_id:'id', mun_nom:'nombre', mun_lat:'lat', mun_lon:'lon', mun_pob:'pob'
})
candidatos = candidatos.rename(columns={
    cand_id:'id', cand_nom:'nombre', cand_lat:'lat', cand_lon:'lon'
})

municipios.head(), candidatos.head()


(   cod_ine                    nombre provincia   pob    latitud  longitud
 0     5007  ALDEANUEVA DE SANTA CRUZ     ÁVILA   102  40.381960 -5.419593
 1     5008                 ALDEASECA     ÁVILA   203  41.049202 -4.817519
 2     5016                   ARÉVALO     ÁVILA  7830  41.065505 -4.718925
 3     5034  BLASCONUÑO DE MATACABRAS     ÁVILA    14  41.123872 -4.989882
 4     5035              BLASCOSANCHO     ÁVILA   100  40.878249 -4.637347,
     id_candidato                                          nombre      tipo  \
 0  AV_SONSESOLES             Hospital Nuestra Señora de Sonsoles  hospital   
 1        BU_HUBU         Hospital Universitario de Burgos (HUBU)  hospital   
 2      LE_HULEON      Complejo Asistencial Universitario de León  hospital   
 3      LE_BIERZO                             Hospital del Bierzo  hospital   
 4       PA_CAUPA  Complejo Asistencial Universitario de Palencia  hospital   
 
     municipio provincia  latitud  longitud  \
 0       Ávila     Ávila  

In [16]:
VEL = 220.0  # km/h
I = municipios['id'].tolist()
J = candidatos['id'].tolist()

t = {}

for _, mu in municipios.iterrows():
    for _, ca in candidatos.iterrows():
        dkm = haversine(mu['lat'], mu['lon'], ca['lat'], ca['lon'])
        tiempo_min = (dkm / VEL) * 60
        t[(mu['id'], ca['id'])] = tiempo_min

t[(I[0], J[0])]


KeyError: 'id'

In [ ]:
!pip install pulp


In [ ]:
import pulp

P = 10  # número de helipuertos a instalar

prob = pulp.LpProblem("p_mediana_cyl", pulp.LpMinimize)

# Variables
x = pulp.LpVariable.dicts("x", J, lowBound=0, upBound=1, cat="Binary")
y = pulp.LpVariable.dicts("y", (I, J), lowBound=0, upBound=1, cat="Binary")

# Objetivo
prob += pulp.lpSum([
    municipios.loc[municipios['id']==i, 'pob'].values[0] * t[(i,j)] * y[i][j]
    for i in I for j in J
])

# Restricciones
for i in I:
    prob += pulp.lpSum(y[i][j] for j in J) == 1

for i in I:
    for j in J:
        prob += y[i][j] <= x[j]

prob += pulp.lpSum(x[j] for j in J) == P


In [ ]:
solver = pulp.PULP_CBC_CMD(msg=True, timeLimit=300)
prob.solve(solver)

print("Estado:", pulp.LpStatus[prob.status])


In [ ]:
selected_centers = [j for j in J if pulp.value(x[j]) == 1]
selected_centers


In [ ]:
asignaciones = []

for i in I:
    for j in J:
        if pulp.value(y[i][j]) == 1:
            asignaciones.append({
                "municipio_id": i,
                "municipio": municipios.loc[municipios["id"]==i, "nombre"].values[0],
                "centro_id": j,
                "centro": candidatos.loc[candidatos["id"]==j, "nombre"].values[0],
                "tiempo_min": t[(i,j)]
            })

asignaciones_df = pd.DataFrame(asignaciones)
asignaciones_df.head()


In [ ]:
asignaciones_df.to_csv("asignaciones_resultado.csv", index=False)
pd.DataFrame(selected_centers, columns=["centro_id"]).to_csv("centros_seleccionados.csv", index=False)

"Archivos guardados correctamente."


In [ ]:
metrics = asignaciones_df.groupby("centro").agg(
    total_poblacion=("tiempo_min", "count"),
    tiempo_medio=("tiempo_min", "mean")
)
metrics
